# Code 5 v2 — Cluster & Commodity Features (Annual-Refresh Aware)

## 📂 INPUT FILES (read from Google Drive)
| File | Source | Notes |
|------|--------|-------|
| `data_features.parquet` | Code 3 | Stock daily Close + features |
| `clusters.csv` | Code 4 v6 | Has `applicable_year` — membership changes yearly |
| `commodities_all.csv` | Code 1d | Incl. USDINR |
| `indices_all.csv` | Code 1c | For Nifty 500 excess returns |

## 📂 OUTPUT FILES (written to /content + auto-downloaded — NOT saved to Drive)
| File | Key | Notes |
|------|-----|-------|
| `cluster_returns.csv` | (applicable_year, cluster, date) | ~52 features incl. excess returns |
| `commodity_returns.csv` | (commodity, date) | ~43 features, NO excess returns, NO year column |

## 🔑 Key Design Decision: Backward-Applied Membership (Q1=Option A)
Cluster membership changes every Jan 1. To compute long-horizon features (e.g. `return_252d`)
**consistently**, each applicable year's membership is applied **backward** over an extended
lookback (~1.5 years before Jan 1). Rolling features are computed on that reconstructed series,
then only rows **within the applicable year** are kept.

- ✅ Each year's features use that year's members consistently — even for lookback
- ✅ No look-ahead: membership decided on past data; returns are past returns
- ⚠️ Documented caveat: features over the formation/lookback window are mildly cohesive
  because the cluster was *selected* for co-movement then (selection bias, not look-ahead)

## Locked Choices
- Equal-weighted aggregation
- 9 returns | 9 excess (clusters only) | 6 vol | 3 sharpe | 3 drawdown | 9 rolling-stat | 4 mom-accel | 9 mean-rev
- `num_stocks` = members **with valid data on that date** (Q4=Option B)
- Nifty 500 forward-filled before excess calc
- Keep NaN at series start (XGBoost handles natively)
- Commodities computed once (no annual refresh); stock→commodity mapping lives in clusters.csv
- Rest + SingleStock clusters included

## 1. Install & Import Libraries

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyarrow', '-q'], check=False)

import pandas as pd
import numpy as np
import os, time, warnings
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.6f}'.format)

print("✓ Libraries ready")

✓ Libraries ready


## 2. Configuration — I/O Manifest + Feature Windows

In [2]:
# ════════════════════════════════════════════════════════════════
# FILE PATHS — UPDATE DRIVE_FOLDER IF YOURS DIFFERS
# ════════════════════════════════════════════════════════════════
DRIVE_FOLDER = '/content/drive/MyDrive/masters'   # 👈 UPDATE if different

# ---- INPUT FILES ----
DATA_FEATURES_FILE = f'{DRIVE_FOLDER}/data_features.parquet'   # Code 3
CLUSTERS_FILE      = f'{DRIVE_FOLDER}/clusters.csv'            # Code 4 v6 (has applicable_year)
COMMODITIES_FILE   = f'{DRIVE_FOLDER}/commodities_all.csv'     # Code 1d
INDICES_FILE       = f'{DRIVE_FOLDER}/indices_all.csv'         # Code 1c

# ---- OUTPUT FILES (LOCAL only — auto-downloaded, NOT saved to Drive) ----
CLUSTER_OUTPUT_FILE   = '/content/cluster_returns.csv'
COMMODITY_OUTPUT_FILE = '/content/commodity_returns.csv'

# Nifty 500 column (None = auto-detect)
NIFTY500_COLUMN = None

# ════════════════════════════════════════════════════════════════
# FEATURE WINDOWS (trading days)
# ════════════════════════════════════════════════════════════════
RETURN_WINDOWS       = [1, 2, 3, 5, 10, 20, 60, 120, 252]   # 9 — matches Code 3
VOL_WINDOWS          = [5, 10, 20, 60, 120, 252]            # skip 1,2 (meaningless)
SHARPE_WINDOWS       = [20, 60, 120]
DRAWDOWN_WINDOWS     = [20, 60, 120]
ROLLING_STAT_WINDOWS = [5, 20, 60]
MEAN_REV_WINDOWS     = [20, 60, 120]
MOMENTUM_PAIRS       = [(5, 20), (20, 60), (60, 120)]
VOL_RATIO_PAIRS      = [(5, 20)]

# Backward lookback buffer (calendar days) before Jan 1 of an applicable year,
# enough to seed the longest window (252 trading days ≈ 1 year → use 1.6y buffer)
LOOKBACK_BUFFER_DAYS = 600

print('═' * 70)
print('  CODE 5 v2 — CONFIGURATION')
print('═' * 70)
print('  INPUTS :', DATA_FEATURES_FILE)
print('          ', CLUSTERS_FILE)
print('          ', COMMODITIES_FILE)
print('          ', INDICES_FILE)
print('  OUTPUTS:', CLUSTER_OUTPUT_FILE)
print('          ', COMMODITY_OUTPUT_FILE)
print('-' * 70)
print(f'  Return windows   : {RETURN_WINDOWS}')
print(f'  Vol windows      : {VOL_WINDOWS}')
print(f'  Lookback buffer  : {LOOKBACK_BUFFER_DAYS} calendar days (seeds 252d window)')
print('═' * 70)

══════════════════════════════════════════════════════════════════════
  CODE 5 v2 — CONFIGURATION
══════════════════════════════════════════════════════════════════════
  INPUTS : /content/drive/MyDrive/masters/data_features.parquet
           /content/drive/MyDrive/masters/clusters.csv
           /content/drive/MyDrive/masters/commodities_all.csv
           /content/drive/MyDrive/masters/indices_all.csv
  OUTPUTS: /content/cluster_returns.csv
           /content/commodity_returns.csv
----------------------------------------------------------------------
  Return windows   : [1, 2, 3, 5, 10, 20, 60, 120, 252]
  Vol windows      : [5, 10, 20, 60, 120, 252]
  Lookback buffer  : 600 calendar days (seeds 252d window)
══════════════════════════════════════════════════════════════════════


## 3. Mount Drive, Verify Inputs, Stage Locally

In [3]:
from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)
print('✓ Drive mounted fresh')

print('\nVerifying inputs...')
print('-' * 70)
ok = True
for label, path in [('data_features.parquet', DATA_FEATURES_FILE),
                    ('clusters.csv',          CLUSTERS_FILE),
                    ('commodities_all.csv',   COMMODITIES_FILE),
                    ('indices_all.csv',       INDICES_FILE)]:
    if os.path.exists(path):
        print(f'  ✓ {label:28s} ({os.path.getsize(path)/1024/1024:.1f} MB)')
    else:
        print(f'  ❌ {label:28s} NOT FOUND: {path}')
        ok = False
if not ok:
    raise FileNotFoundError('Missing inputs — fix paths in Section 2.')

LOCAL = '/content/inputs'
os.makedirs(LOCAL, exist_ok=True)
import shutil
LOCAL_FEATURES    = f'{LOCAL}/data_features.parquet'
LOCAL_CLUSTERS    = f'{LOCAL}/clusters.csv'
LOCAL_COMMODITIES = f'{LOCAL}/commodities_all.csv'
LOCAL_INDICES     = f'{LOCAL}/indices_all.csv'

print('\nStaging Drive → local...')
for src, dst in [(DATA_FEATURES_FILE, LOCAL_FEATURES), (CLUSTERS_FILE, LOCAL_CLUSTERS),
                 (COMMODITIES_FILE, LOCAL_COMMODITIES), (INDICES_FILE, LOCAL_INDICES)]:
    if not os.path.exists(dst):
        shutil.copy(src, dst)
    print(f'  ✓ {os.path.basename(dst)}')
print('\n✓ Inputs staged locally')

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
✓ Drive mounted fresh

Verifying inputs...
----------------------------------------------------------------------
  ✓ data_features.parquet        (1303.9 MB)
  ✓ clusters.csv                 (0.5 MB)
  ✓ commodities_all.csv          (1.2 MB)
  ✓ indices_all.csv              (1.1 MB)

Staging Drive → local...
  ✓ data_features.parquet
  ✓ clusters.csv
  ✓ commodities_all.csv
  ✓ indices_all.csv

✓ Inputs staged locally


## 4. Load Inputs

In [4]:
# ── Stock features ──────────────────────────────────────────────
print('Loading stock features...')
df = pd.read_parquet(LOCAL_FEATURES)
rename_map = {}
for c in df.columns:
    cl = c.lower()
    if cl == 'date':                  rename_map[c] = 'date'
    elif cl in ('ticker', 'symbol'):  rename_map[c] = 'symbol'
    elif cl == 'sector':              rename_map[c] = 'sector'
    elif cl == 'close':               rename_map[c] = 'Close'
    elif cl == 'adj close':           rename_map[c] = 'Close'
df = df.rename(columns=rename_map)
df['date'] = pd.to_datetime(df['date'])
close_col = next((c for c in ['Close','Adj Close','adj_close','close'] if c in df.columns), None)
if close_col is None:
    raise ValueError('No Close price column found')
print(f'  Rows {len(df):,} | Stocks {df["symbol"].nunique():,} | '
      f'{df["date"].min().date()} → {df["date"].max().date()} | Close="{close_col}"')

# ── Clusters (annual) ───────────────────────────────────────────
print('\nLoading clusters (annual-refresh)...')
df_clusters = pd.read_csv(LOCAL_CLUSTERS)
if 'applicable_year' not in df_clusters.columns:
    raise ValueError("clusters.csv missing 'applicable_year' — is this Code 4 v6 output?")
print(f'  Rows {len(df_clusters):,} | Years {sorted(df_clusters["applicable_year"].unique())}')
print(f'  Unique clusters across years: {df_clusters["cluster"].nunique()}')

# ── Commodities ─────────────────────────────────────────────────
print('\nLoading commodities...')
df_comm = pd.read_csv(LOCAL_COMMODITIES)
for c in ('Date','date'):
    if c in df_comm.columns:
        df_comm = df_comm.rename(columns={c:'date'}); break
df_comm['date'] = pd.to_datetime(df_comm['date'])
df_comm = df_comm.sort_values('date').reset_index(drop=True)
COMMODITY_COLS = [c for c in df_comm.columns if c != 'date']
print(f'  Rows {len(df_comm):,} | Commodities ({len(COMMODITY_COLS)}): {COMMODITY_COLS}')

# ── Indices + Nifty 500 detection ───────────────────────────────
print('\nLoading indices...')
df_idx = pd.read_csv(LOCAL_INDICES)
for c in ('Date','date'):
    if c in df_idx.columns:
        df_idx = df_idx.rename(columns={c:'date'}); break
df_idx['date'] = pd.to_datetime(df_idx['date'])
df_idx = df_idx.sort_values('date').reset_index(drop=True)
INDEX_COLS = [c for c in df_idx.columns if c != 'date']

if NIFTY500_COLUMN is None:
    cands = [c for c in INDEX_COLS
             if 'NIFTY500' in c.upper().replace(' ','').replace('_','').replace('-','')
             or 'CNX500' in c.upper().replace(' ','') or '^CRSLDX' in c.upper()]
    if not cands:
        raise ValueError(f'No Nifty 500 column found in {INDEX_COLS}. Set NIFTY500_COLUMN.')
    NIFTY500 = cands[0]
    print(f'  ✓ Auto-detected Nifty 500: {NIFTY500}' +
          (f'  (also {cands[1:]})' if len(cands) > 1 else ''))
else:
    NIFTY500 = NIFTY500_COLUMN
    print(f'  ✓ Using Nifty 500: {NIFTY500}')

Loading stock features...
  Rows 2,417,660 | Stocks 570 | 2007-01-02 → 2026-06-12 | Close="Close"

Loading clusters (annual-refresh)...
  Rows 7,766 | Years [np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
  Unique clusters across years: 277

Loading commodities...
  Rows 5,070 | Commodities (17): ['GOLD', 'SILVER', 'PLATINUM', 'PALLADIUM', 'COPPER', 'ALUMINIUM', 'CRUDE_OIL_WTI', 'BRENT_CRUDE', 'NATURAL_GAS', 'SUGAR', 'COTTON', 'SOYBEAN', 'WHEAT', 'CORN', 'COFFEE', 'COCOA', 'USDINR']

Loading indices...
  ✓ Auto-detected Nifty 500: NIFTY_500


## 5. Stock Daily Returns + Wide Price Pivot

We need a date×symbol matrix of daily returns to reconstruct cluster series quickly.

In [5]:
print('Computing stock daily returns...')
df = df.sort_values(['symbol','date']).reset_index(drop=True)
df['stock_return_1d'] = df.groupby('symbol')[close_col].pct_change()

# Wide matrix: date × symbol of daily returns (used for fast cluster reconstruction)
ret_wide = df.pivot_table(index='date', columns='symbol',
                          values='stock_return_1d', aggfunc='first').sort_index()
all_trading_dates = ret_wide.index
print(f'  Return matrix: {ret_wide.shape} (dates × stocks)')
print(f'  Trading dates: {all_trading_dates.min().date()} → {all_trading_dates.max().date()}')

Computing stock daily returns...
  Return matrix: (4798, 570) (dates × stocks)
  Trading dates: 2007-01-03 → 2026-06-12


## 6. Nifty 500 Daily + Multi-Horizon Returns (Forward-Filled)

In [6]:
print('Building Nifty 500 returns (forward-filled to NSE dates)...')
nse_dates = pd.DataFrame({'date': all_trading_dates})
nifty = df_idx[['date', NIFTY500]].rename(columns={NIFTY500:'nifty500'}).sort_values('date')
nifty = nse_dates.merge(nifty, on='date', how='left')
miss_before = nifty['nifty500'].isna().sum()
nifty['nifty500'] = nifty['nifty500'].ffill()
print(f'  Forward-filled {miss_before} missing values; '
      f'{nifty["nifty500"].isna().sum()} remain (pre-history start)')

# Multi-horizon Nifty 500 returns (as fraction; cluster returns are also fractions)
nifty = nifty.set_index('date')
for w in RETURN_WINDOWS:
    nifty[f'nifty500_return_{w}d'] = nifty['nifty500'].pct_change(w)
print(f'  ✓ Nifty 500 returns at {len(RETURN_WINDOWS)} horizons')

Building Nifty 500 returns (forward-filled to NSE dates)...
  Forward-filled 12 missing values; 0 remain (pre-history start)
  ✓ Nifty 500 returns at 9 horizons


## 7. Feature-Engineering Helper (Shared by Clusters & Commodities)

Operates on a single daily-return Series indexed by date. All windows in trading days.

In [7]:
def compound_return(s, w):
    return (1 + s).rolling(w, min_periods=max(1, w // 2)).apply(lambda y: y.prod() - 1, raw=True)

def rolling_mdd(returns):
    cum = (1 + returns).cumprod()
    return ((cum / cum.cummax()) - 1).min()

def build_features(daily, with_excess=False, nifty_df=None):
    """daily: pd.Series of 1d returns indexed by date.
    Returns a DataFrame (same index) with all engineered features."""
    s = daily
    out = {'return_1d': s}

    # Compound returns
    for w in RETURN_WINDOWS:
        out[f'return_{w}d'] = compound_return(s, w)

    # Volatility
    for w in VOL_WINDOWS:
        out[f'volatility_{w}d'] = s.rolling(w, min_periods=max(1, w//2)).std()

    # Sharpe (annualized, rf=0)
    for w in SHARPE_WINDOWS:
        m = s.rolling(w, min_periods=max(1, w//2)).mean()
        sd = s.rolling(w, min_periods=max(1, w//2)).std()
        out[f'sharpe_{w}d'] = (m * np.sqrt(252)) / (sd * np.sqrt(252) + 1e-9)

    # Max drawdown
    for w in DRAWDOWN_WINDOWS:
        out[f'max_drawdown_{w}d'] = s.rolling(w, min_periods=max(1, w//2)).apply(rolling_mdd, raw=False)

    # Rolling stats
    for w in ROLLING_STAT_WINDOWS:
        out[f'max_return_{w}d'] = s.rolling(w, min_periods=max(1, w//2)).max()
        out[f'min_return_{w}d'] = s.rolling(w, min_periods=max(1, w//2)).min()
        out[f'positive_days_ratio_{w}d'] = (s > 0).rolling(w, min_periods=max(1, w//2)).mean()

    # Momentum acceleration
    for a, b in MOMENTUM_PAIRS:
        out[f'momentum_accel_{a}_{b}'] = out[f'return_{a}d'] - out[f'return_{b}d']
    for a, b in VOL_RATIO_PAIRS:
        out[f'vol_ratio_{a}_{b}'] = out[f'volatility_{a}d'] / (out[f'volatility_{b}d'] + 1e-9)

    # Mean reversion
    cum = (1 + s.fillna(0)).cumprod()
    for w in MEAN_REV_WINDOWS:
        m = s.rolling(w, min_periods=max(1, w//2)).mean()
        sd = s.rolling(w, min_periods=max(1, w//2)).std()
        out[f'z_score_{w}d'] = (s - m) / (sd + 1e-9)
        rmax = cum.rolling(w, min_periods=max(1, w//2)).max()
        rmin = cum.rolling(w, min_periods=max(1, w//2)).min()
        out[f'distance_from_high_{w}d'] = (cum - rmax) / (rmax + 1e-9)
        out[f'distance_from_low_{w}d']  = (cum - rmin) / (rmin + 1e-9)

    feat = pd.DataFrame(out, index=s.index)

    # Excess returns vs Nifty 500 (clusters only)
    if with_excess and nifty_df is not None:
        for w in RETURN_WINDOWS:
            nser = nifty_df[f'nifty500_return_{w}d'].reindex(feat.index)
            feat[f'excess_return_{w}d'] = feat[f'return_{w}d'] - nser
    return feat

print('✓ Feature helper defined')

✓ Feature helper defined


## 8. CLUSTER FEATURES — Backward-Applied Membership (Option A)

For each (applicable_year, cluster):
1. Take that year's members
2. Slice daily returns over [Jan1(year) − buffer  →  Dec31(year)]
3. Equal-weight average across members per date (reconstructed series)
4. Compute all features on the reconstructed series
5. Keep only rows **within the applicable year**
6. `num_stocks` = members with valid data on each date (Option B)

In [8]:
t0 = time.time()

# Map: applicable_year → {cluster_name → [members]}
year_cluster_members = {}
for (yr, cl), grp in df_clusters.groupby(['applicable_year', 'cluster']):
    year_cluster_members.setdefault(yr, {})[cl] = grp['symbol'].tolist()

years_sorted = sorted(year_cluster_members.keys())
print(f'Building cluster features for {len(years_sorted)} years '
      f'({years_sorted[0]}–{years_sorted[-1]})...')

cluster_frames = []
total_units = sum(len(v) for v in year_cluster_members.values())
done = 0

for yr in years_sorted:
    members_map = year_cluster_members[yr]
    yr_start = pd.Timestamp(f'{yr}-01-01')
    yr_end   = pd.Timestamp(f'{yr}-12-31')
    lookback_start = yr_start - pd.Timedelta(days=LOOKBACK_BUFFER_DAYS)

    # Dates available in the reconstruction window
    win_mask = (all_trading_dates >= lookback_start) & (all_trading_dates <= yr_end)
    win_dates = all_trading_dates[win_mask]

    for cl, members in members_map.items():
        done += 1
        members_present = [m for m in members if m in ret_wide.columns]
        if not members_present:
            continue

        # Reconstruct equal-weighted daily return series over lookback+year
        sub = ret_wide.loc[win_dates, members_present]
        cluster_daily = sub.mean(axis=1, skipna=True)            # equal-weighted
        num_stocks = sub.notna().sum(axis=1)                     # valid members/date (Option B)

        # Features over full reconstructed series
        feat = build_features(cluster_daily, with_excess=True, nifty_df=nifty)

        # Keep only rows within the applicable year
        keep = (feat.index >= yr_start) & (feat.index <= yr_end)
        feat = feat.loc[keep].copy()
        if feat.empty:
            continue
        feat.insert(0, 'applicable_year', yr)
        feat.insert(1, 'cluster', cl)
        feat.insert(2, 'date', feat.index)
        feat.insert(3, 'num_stocks', num_stocks.reindex(feat.index).values)
        cluster_frames.append(feat.reset_index(drop=True))

    if yr % 3 == 0 or yr == years_sorted[-1]:
        print(f'  {yr}: {done}/{total_units} cluster-years  ({time.time()-t0:.1f}s)')

cluster_features = pd.concat(cluster_frames, ignore_index=True)
print(f'\n✓ Cluster features: {cluster_features.shape}  in {time.time()-t0:.1f}s')
print(f'  Columns: {len(cluster_features.columns)}')

Building cluster features for 15 years (2011–2025)...
  2013: 150/836 cluster-years  (109.5s)
  2016: 279/836 cluster-years  (207.0s)
  2019: 442/836 cluster-years  (328.5s)
  2022: 627/836 cluster-years  (465.0s)
  2025: 836/836 cluster-years  (620.2s)

✓ Cluster features: (206229, 56)  in 620.4s
  Columns: 56


## 9. COMMODITY FEATURES — Computed Once (No Annual Refresh, No Excess)

In [9]:
t0 = time.time()
print(f'Building commodity features for {len(COMMODITY_COLS)} commodities...')

comm_frames = []
for cm in COMMODITY_COLS:
    daily = df_comm.set_index('date')[cm].pct_change()
    feat = build_features(daily, with_excess=False)
    feat.insert(0, 'commodity', cm)
    feat.insert(1, 'date', feat.index)
    comm_frames.append(feat.reset_index(drop=True))

commodity_features = pd.concat(comm_frames, ignore_index=True)
print(f'✓ Commodity features: {commodity_features.shape}  in {time.time()-t0:.1f}s')
print(f'  Columns: {len(commodity_features.columns)}  (no excess_return, no applicable_year)')

Building commodity features for 17 commodities...
✓ Commodity features: (86190, 45)  in 112.5s
  Columns: 45  (no excess_return, no applicable_year)


## 10. Data Quality Checks

In [10]:
def quality(dfx, name, key_cols):
    print(f'\n{"═"*70}\n  {name}\n{"═"*70}')
    print(f'  Shape      : {dfx.shape}')
    print(f'  Date range : {dfx["date"].min().date()} → {dfx["date"].max().date()}')
    for k in key_cols:
        if k in dfx.columns:
            print(f'  {k:16s}: {dfx[k].nunique()} unique')
    nan = dfx.isna().sum().sort_values(ascending=False)
    nan = nan[nan > 0].head(12)
    print('  Top NaN columns:' if len(nan) else '  No NaN values')
    for c, n in nan.items():
        print(f'    {c:34s} {n:>8,} ({n/len(dfx)*100:4.1f}%)')
    show = [c for c in ['return_5d','return_252d','volatility_20d','sharpe_20d',
                        'max_drawdown_60d','z_score_20d','excess_return_5d'] if c in dfx.columns]
    print('  Sample stats:')
    print(dfx[show].describe().loc[['mean','std','min','max']].to_string())

quality(cluster_features,   'CLUSTER FEATURES',   ['applicable_year','cluster'])
quality(commodity_features, 'COMMODITY FEATURES', ['commodity'])

# Boundary continuity spot-check: same cluster name across two consecutive years
print('\n' + '─'*70)
print('Boundary spot-check (membership change across Jan 1):')
common_names = (cluster_features.groupby('cluster')['applicable_year']
                .nunique().sort_values(ascending=False))
multi_year = common_names[common_names > 1].index[:1]
if len(multi_year):
    nm = multi_year[0]
    sub = cluster_features[cluster_features['cluster'] == nm]
    yrs = sorted(sub['applicable_year'].unique())[:2]
    print(f'  Cluster "{nm}" appears in years {sorted(sub["applicable_year"].unique())}')
    for y in yrs:
        mem = year_cluster_members[y][nm]
        print(f'    {y}: {len(mem)} members — {[m.replace(".NS","") for m in mem[:6]]}'
              + (' ...' if len(mem) > 6 else ''))


══════════════════════════════════════════════════════════════════════
  CLUSTER FEATURES
══════════════════════════════════════════════════════════════════════
  Shape      : (206229, 56)
  Date range : 2011-01-03 → 2025-12-31
  applicable_year : 15 unique
  cluster         : 277 unique
  Top NaN columns:
    return_252d                          11,340 ( 5.5%)
    excess_return_252d                   11,340 ( 5.5%)
    momentum_accel_60_120                 4,727 ( 2.3%)
    return_120d                           4,727 ( 2.3%)
    excess_return_120d                    4,727 ( 2.3%)
    momentum_accel_20_60                  2,387 ( 1.2%)
    return_60d                            2,387 ( 1.2%)
    excess_return_60d                     2,387 ( 1.2%)
    excess_return_20d                       800 ( 0.4%)
    momentum_accel_5_20                     800 ( 0.4%)
    return_20d                              800 ( 0.4%)
    excess_return_10d                       400 ( 0.2%)
  Sample stats:
   

## 11. Sample Output

In [11]:
print('Cluster features — sample (last rows of one cluster-year):')
nm = cluster_features['cluster'].mode().iloc[0]
yr = cluster_features[cluster_features['cluster']==nm]['applicable_year'].max()
samp = cluster_features[(cluster_features['cluster']==nm) &
                        (cluster_features['applicable_year']==yr)].tail(6)
display(samp[['applicable_year','cluster','date','num_stocks','return_5d',
              'excess_return_5d','return_20d','volatility_20d','sharpe_20d','z_score_20d']])

print('\nCommodity features — sample:')
cm = 'GOLD' if 'GOLD' in commodity_features['commodity'].values else commodity_features['commodity'].iloc[0]
samp = commodity_features[commodity_features['commodity']==cm].tail(6)
display(samp[['commodity','date','return_5d','return_20d','volatility_20d',
              'sharpe_20d','max_drawdown_60d','z_score_20d']])

Cluster features — sample (last rows of one cluster-year):


,applicable_year,cluster,date,num_stocks,return_5d,excess_return_5d,return_20d,volatility_20d,sharpe_20d,z_score_20d
189042,2025,BasicMaterials-ACC,2025-12-23,2,0.012429,-0.001359,-0.014247,0.010125,-0.066058,2.958530
189043,2025,BasicMaterials-ACC,2025-12-24,2,-0.002020,-0.017095,-0.027641,0.011121,-0.120675,-1.916950
189044,2025,BasicMaterials-ACC,2025-12-26,2,0.004992,-0.007300,-0.040793,0.010769,-0.188071,-0.106090
189045,2025,BasicMaterials-ACC,2025-12-29,2,0.009403,0.009211,-0.028073,0.010712,-0.127745,0.499802
189046,2025,BasicMaterials-ACC,2025-12-30,2,0.000959,0.009290,-0.036510,0.010705,-0.168493,-0.371020
189047,2025,BasicMaterials-ACC,2025-12-31,2,-0.025952,-0.024965,-0.024682,0.010527,-0.113648,0.267026



Commodity features — sample:


,commodity,date,return_5d,return_20d,volatility_20d,sharpe_20d,max_drawdown_60d,z_score_20d
5064,GOLD,2026-06-05,-0.048986,-0.081201,0.011737,-0.354366,-0.132753,-2.285794
5065,GOLD,2026-06-08,-0.031127,-0.081124,0.011739,-0.353968,-0.132993,0.330394
5066,GOLD,2026-06-09,-0.051035,-0.089277,0.012078,-0.380428,-0.128817,-1.068860
5067,GOLD,2026-06-10,-0.074042,-0.125487,0.013720,-0.480434,-0.154274,-2.116806
5068,GOLD,2026-06-11,-0.086130,-0.125649,0.013718,-0.481166,-0.157959,0.163546
5069,GOLD,2026-06-12,-0.028152,-0.074806,0.015233,-0.247439,-0.157959,2.248806


## 12. Save Outputs Locally + Auto-Download (no Drive write)

In [12]:
cluster_features = cluster_features.sort_values(
    ['applicable_year','cluster','date']).reset_index(drop=True)
commodity_features = commodity_features.sort_values(
    ['commodity','date']).reset_index(drop=True)

# Write to LOCAL Colab storage only (NOT Drive) — Drive is input-only
print('Writing output files locally (/content)...')
cluster_features.to_csv(CLUSTER_OUTPUT_FILE, index=False)
print(f'  ✓ {CLUSTER_OUTPUT_FILE}')
print(f'    {len(cluster_features):,} rows × {len(cluster_features.columns)} cols, '
      f'{os.path.getsize(CLUSTER_OUTPUT_FILE)/1024/1024:.1f} MB')
commodity_features.to_csv(COMMODITY_OUTPUT_FILE, index=False)
print(f'  ✓ {COMMODITY_OUTPUT_FILE}')
print(f'    {len(commodity_features):,} rows × {len(commodity_features.columns)} cols, '
      f'{os.path.getsize(COMMODITY_OUTPUT_FILE)/1024/1024:.1f} MB')

# Auto-download to your computer's Downloads folder
print('\nDownloading to your computer (check Downloads folder)...')
from google.colab import files
for p in [CLUSTER_OUTPUT_FILE, COMMODITY_OUTPUT_FILE]:
    try:
        files.download(p); print(f'  ✓ {os.path.basename(p)}')
    except Exception as e:
        print(f'  ⚠ Could not download {os.path.basename(p)}: {e}')
        print(f'    File is still at {p} — download manually from the Files panel.')

Writing output files locally (/content)...
  ✓ /content/cluster_returns.csv
    206,229 rows × 56 cols, 208.0 MB
  ✓ /content/commodity_returns.csv
    86,190 rows × 45 cols, 68.6 MB



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ✓ cluster_returns.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ✓ commodity_returns.csv


## 13. Final Summary

In [13]:
print('═' * 70)
print('  CODE 5 v2 — COMPLETE')
print('═' * 70)
print(f'  cluster_returns.csv   : {len(cluster_features):,} rows × {len(cluster_features.columns)} cols')
print(f'    key = (applicable_year, cluster, date)')
print(f'    years = {cluster_features["applicable_year"].min()}–{cluster_features["applicable_year"].max()}')
print(f'  commodity_returns.csv : {len(commodity_features):,} rows × {len(commodity_features.columns)} cols')
print(f'    key = (commodity, date)')
print()
print('  Membership: backward-applied per year (Option A) — long-horizon features consistent')
print('  Excess returns: clusters only, vs forward-filled Nifty 500')
print('  num_stocks: valid members per date (Option B)')
print()
print('  Next → Code 6: ATR-based exit labels (High/Medium/Low/Ignore)')
print('         Code 7 will join: stock features + cluster features (on symbol+year via')
print('         clusters.csv) + mapped-commodity features + USDINR features')
print('═' * 70)

══════════════════════════════════════════════════════════════════════
  CODE 5 v2 — COMPLETE
══════════════════════════════════════════════════════════════════════
  cluster_returns.csv   : 206,229 rows × 56 cols
    key = (applicable_year, cluster, date)
    years = 2011–2025
  commodity_returns.csv : 86,190 rows × 45 cols
    key = (commodity, date)

  Membership: backward-applied per year (Option A) — long-horizon features consistent
  Excess returns: clusters only, vs forward-filled Nifty 500
  num_stocks: valid members per date (Option B)

  Next → Code 6: ATR-based exit labels (High/Medium/Low/Ignore)
         Code 7 will join: stock features + cluster features (on symbol+year via
         clusters.csv) + mapped-commodity features + USDINR features
══════════════════════════════════════════════════════════════════════
